# Loading Data

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamkushwaha/bbc-full-text-document-classification")

print("Path to dataset files:", path)

100%|██████████| 5.59M/5.59M [00:00<00:00, 73.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1


# Import Necessary Dependencies

In [2]:
import numpy as np
import pandas as pd

# Convert Text Data to CSV

In [3]:
# show folders from this path : /root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1
import os
for dirname in os.walk('/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1'):
        print(dirname)

('/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1', ['bbc-fulltext (document classification)', 'bbc'], [])
('/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1/bbc-fulltext (document classification)', ['bbc'], [])
('/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1/bbc-fulltext (document classification)/bbc', ['business', 'sport', 'politics', 'entertainment', 'tech'], ['README.TXT'])
('/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1/bbc-fulltext (document classification)/bbc/business', [], ['412.txt', '311.txt', '365.txt', '107.txt', '304.txt', '110.txt', '020.txt', '482.txt', '320.txt', '508.txt', '220.txt', '054.txt', '441.txt', '028.txt', '378.txt', '323.txt', '192.txt', '001.txt', '427.txt', '421.txt', '084.txt', '219.txt', '339.txt', '356.txt', '341.txt', '437.txt', '343.txt', '058.txt', '353.txt'

In [4]:
import os
import pandas as pd

DATASET_PATH = "/root/.cache/kagglehub/datasets/shivamkushwaha/bbc-full-text-document-classification/versions/1/bbc"

rows = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for file in os.listdir(category_path):

        if not file.endswith(".txt"):
            continue

        file_path = os.path.join(category_path, file)

        try:
            with open(file_path, "r", encoding="latin-1") as f:
                text = f.read()

            rows.append({
                "category": category,
                "filename": file,
                "text": text
            })

        except Exception as e:
            print(f"Error reading {file}: {e}")

df = pd.DataFrame(rows)

print(df.shape)

df.to_csv(
    "bbc_news_dataset.csv",
    index=False
)

print("Saved: bbc_news_dataset.csv")

(2225, 3)
Saved: bbc_news_dataset.csv


In [5]:
news_csv = pd.read_csv("bbc_news_dataset.csv")

In [6]:
news_csv.head(5)

,category,filename,text
0,business,412.txt,Iran budget seeks state sell-offs\n\nIran's pr...
1,business,311.txt,Stormy year for property insurers\n\nA string ...
2,business,365.txt,Nasdaq planning $100m-share sale\n\nThe owner ...
3,business,107.txt,Liberian economy starts to grow\n\nThe Liberia...
4,business,304.txt,Bombardier chief to leave company\n\nShares in...


In [7]:
pd.unique(df.category)

array(['business', 'sport', 'politics', 'entertainment', 'tech'],
      dtype=object)

# Data PreProcessing

- this step includes, removing null values, duplicated values, text cleaning, stopword removal, lemmetization

#### Null Values

In [8]:
pd.isnull(df.category).sum()

np.int64(0)

In [9]:
pd.isnull(df.filename).sum()

np.int64(0)

In [10]:
pd.isnull(df.text).sum()

np.int64(0)

No null values exists

#### Duplicate Values

In [11]:
df.duplicated().sum()

np.int64(0)

no duplicate rows

### Text cleaning (removing punctuations, numbers & special characters)

In [12]:
import re


def clean_text(text):
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]*>', '', text)

    # 2. Remove URLs / Hyperlinks
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Remove Email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # 4. Remove everything except letters and numbers (keeps spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # 5. Remove numbers/digits
    text = re.sub(r'\d+', '', text)

    # 6. Replace multiple spaces/newlines with a single space
    text = re.sub(r'\s+', ' ', text)

    # 7. Strip leading and trailing whitespace and lowercase
    return text.strip().lower()

In [13]:
news_csv['clean_text'] = news_csv['text'].apply(lambda x: clean_text(x))

In [14]:
news_csv.head(6)

,category,filename,text,clean_text
0,business,412.txt,Iran budget seeks state sell-offs\n\nIran's pr...,iran budget seeks state selloffs irans preside...
1,business,311.txt,Stormy year for property insurers\n\nA string ...,stormy year for property insurers a string of ...
2,business,365.txt,Nasdaq planning $100m-share sale\n\nThe owner ...,nasdaq planning mshare sale the owner of the t...
3,business,107.txt,Liberian economy starts to grow\n\nThe Liberia...,liberian economy starts to grow the liberian e...
4,business,304.txt,Bombardier chief to leave company\n\nShares in...,bombardier chief to leave company shares in tr...
5,business,110.txt,Japanese growth grinds to a halt\n\nGrowth in ...,japanese growth grinds to a halt growth in jap...


### removing stopwords

In [15]:
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [16]:
# remove stopwords

stop_words = set(stopwords.words('english'))

In [17]:
custom_stopwords = {
    "said", "would", "could", "also",
    "say", "told", "one", "new",
    "year", "years", "week",
    "last", "next", "may",
    "make", "made", "added",
    "bbc", "people", "time"
}

In [18]:
news_csv['clean_text'] = news_csv['clean_text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

In [19]:
news_csv['clean_text'] = news_csv['clean_text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (custom_stopwords)]))

In [20]:
news_csv['clean_text'].head(1).tolist()

['iran budget seeks state selloffs irans president mohammad khatami unveiled budget designed expand public spending loosen islamic republics dependence oil budget fiscal starting march calls selloff states corporate holdings mr khatamis second term president ends august making budget opposition members parliament attacked previous privatisations block plans elections ousted many mr khatamis supporters parliament favour hardline religious conservatives late backed law give parliament veto foreign investment ruling response involvement telecoms airport projects turkish companies hardliners accused business israel came long expediency council irans ultimate decisionmaker blessed mr khatamis policy selling stakes sectors protected constitution energy transport telecoms banking continued obstruction foreign investment get way privatisation plans mr khatamis hope modestly reducing governments reliance oil revenues address majlis mr khatami predicted economic growth current wanted increase bu

### removing short words (noise)

In [21]:
tokens = news_csv['clean_text'].apply(
    lambda words: [w for w in words.split() if len(w) > 2]
)

In [22]:
news_csv['tokens'] = tokens

In [23]:
news_csv.head()

,category,filename,text,clean_text,tokens
0,business,412.txt,Iran budget seeks state sell-offs\n\nIran's pr...,iran budget seeks state selloffs irans preside...,"[iran, budget, seeks, state, selloffs, irans, ..."
1,business,311.txt,Stormy year for property insurers\n\nA string ...,stormy property insurers string storms typhoon...,"[stormy, property, insurers, string, storms, t..."
2,business,365.txt,Nasdaq planning $100m-share sale\n\nThe owner ...,nasdaq planning mshare sale owner technologydo...,"[nasdaq, planning, mshare, sale, owner, techno..."
3,business,107.txt,Liberian economy starts to grow\n\nThe Liberia...,liberian economy starts grow liberian economy ...,"[liberian, economy, starts, grow, liberian, ec..."
4,business,304.txt,Bombardier chief to leave company\n\nShares in...,bombardier chief leave company shares train pl...,"[bombardier, chief, leave, company, shares, tr..."


In [24]:
news_csv['tokens'].apply(len).describe()

,tokens
count,2225.000000
mean,196.986067
std,115.250370
min,43.000000
25%,128.000000
50%,172.000000
75%,241.000000
max,2034.000000


### lemmetization using nltk

In [25]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

this process does:
playing -> play,
running -> run

convert the word into base form

In [26]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

news_csv["lemmas"] = news_csv["tokens"].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

### converting list to string

In [27]:
news_csv['lda_text'] = news_csv['lemmas'].apply(
    lambda x: ' '.join(x)
)

In [28]:
news_csv.head()

,category,filename,text,clean_text,tokens,lemmas,lda_text
0,business,412.txt,Iran budget seeks state sell-offs\n\nIran's pr...,iran budget seeks state selloffs irans preside...,"[iran, budget, seeks, state, selloffs, irans, ...","[iran, budget, seek, state, selloff, iran, pre...",iran budget seek state selloff iran president ...
1,business,311.txt,Stormy year for property insurers\n\nA string ...,stormy property insurers string storms typhoon...,"[stormy, property, insurers, string, storms, t...","[stormy, property, insurer, string, storm, typ...",stormy property insurer string storm typhoon e...
2,business,365.txt,Nasdaq planning $100m-share sale\n\nThe owner ...,nasdaq planning mshare sale owner technologydo...,"[nasdaq, planning, mshare, sale, owner, techno...","[nasdaq, planning, mshare, sale, owner, techno...",nasdaq planning mshare sale owner technologydo...
3,business,107.txt,Liberian economy starts to grow\n\nThe Liberia...,liberian economy starts grow liberian economy ...,"[liberian, economy, starts, grow, liberian, ec...","[liberian, economy, start, grow, liberian, eco...",liberian economy start grow liberian economy s...
4,business,304.txt,Bombardier chief to leave company\n\nShares in...,bombardier chief leave company shares train pl...,"[bombardier, chief, leave, company, shares, tr...","[bombardier, chief, leave, company, share, tra...",bombardier chief leave company share train pla...


# Vectorization

## Word Vectorization to check similarities

### Word2Vec, TfIdf, countvectorizer

-------------------------------------

r1: [car automobile vehicle]
r2: [car driving]

this 3 are unrelated words [Logically they're related for humans, but for mathematics it isn't]

-------------------------------------
countvectorizer =>
r1: [1 1 1]
r2: [2 1 0]
It counts the words but not understand meaning.

-------------------------------------
TFIDF => Term freq * Inverse Document freq

eg. government (appears only in political docs, gets high importance)

eg. said (appears everywhere, gets lowest importance), but still car != automobile
Good for similarity.

-------------------------------------

Word2Vec => Learns vector representation from context,

eg. r1[car automobile vehicle]

eg vector conversion.
r1[[0.12, 0.8, 0.1...][0.11, 0.7, -0.1...][0.13, 0.7, -0.5...]

Although r1 has 3 unknown words, but contextually their vectors will be nearby, so when we make clusters it appears so well.

-------------------------------------
decision: Word2Vec
-------------------------------------


In [29]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.0 MB/s eta 0:00:00


In [30]:
# Using Lemmetized words to create vectors

from gensim.models import Word2Vec

sentences = news_csv["lemmas"].tolist()

# training word2vec model
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=1  # Skip-Gram
)

In [31]:
# Evaluation in simple terms

w2v_model.wv.most_similar("market")

[('stock', 0.7214646935462952),
 ('manufacturer', 0.6844290494918823),
 ('investor', 0.6776865124702454),
 ('housing', 0.6765967011451721),
 ('trend', 0.6745057702064514),
 ('exchange', 0.6641865968704224),
 ('jupiter', 0.6538578867912292),
 ('europe', 0.6492794156074524),
 ('gartner', 0.6448674201965332),
 ('demand', 0.6434873938560486)]

interpretation: news generally have similar topics like a stock market news, manufacturing of something, house selling market rate etc...

taking similar examples to check generalization

In [32]:
w2v_model.wv.most_similar("trend")

[('manufacturer', 0.959890604019165),
 ('grow', 0.9562829732894897),
 ('booming', 0.9358765482902527),
 ('improving', 0.9331409931182861),
 ('continues', 0.9321044087409973),
 ('driven', 0.9312752485275269),
 ('popularity', 0.9254755973815918),
 ('growing', 0.9246687889099121),
 ('predicts', 0.9233679175376892),
 ('improvement', 0.9209932684898376)]

In [33]:
w2v_model.wv.most_similar("stock")

[('exchange', 0.8891773819923401),
 ('share', 0.885711133480072),
 ('trading', 0.8642793297767639),
 ('investor', 0.853985607624054),
 ('nasdaq', 0.8503184914588928),
 ('listing', 0.8412228226661682),
 ('penny', 0.8241620659828186),
 ('closed', 0.8195772767066956),
 ('cairn', 0.7939420342445374),
 ('capital', 0.7844090461730957)]

In [34]:
print(f"Company: {w2v_model.wv.most_similar("company")} \n profit: {w2v_model.wv.most_similar("profit")
} \n Bank: {w2v_model.wv.most_similar("bank")}")

Company: [('firm', 0.8501873016357422), ('giant', 0.7452182769775391), ('asset', 0.7347562909126282), ('unit', 0.7290835976600647), ('fiat', 0.7268470525741577), ('gas', 0.719517171382904), ('business', 0.7190399169921875), ('bought', 0.7170504927635193), ('gazprom', 0.716022253036499), ('shareholder', 0.7153484225273132)] 
 profit: [('revenue', 0.9250878691673279), ('earnings', 0.9091435074806213), ('euro', 0.8829324841499329), ('steel', 0.8657204508781433), ('pretax', 0.8538036346435547), ('posted', 0.8520843386650085), ('risen', 0.8453477025032043), ('rose', 0.8417691588401794), ('fuel', 0.8415409922599792), ('worldwide', 0.8330610990524292)] 
 Bank: [('loan', 0.7294816970825195), ('reserve', 0.7217724323272705), ('monetary', 0.6935746073722839), ('central', 0.6884835958480835), ('account', 0.6841117143630981), ('lender', 0.6828210353851318), ('ratesetting', 0.6623970866203308), ('capital', 0.6621193885803223), ('china', 0.6608009338378906), ('raised', 0.6598235964775085)]


In [35]:
print(w2v_model.wv.most_similar("government"))
print(w2v_model.wv.most_similar("player"))
print(w2v_model.wv.most_similar("software"))
print(w2v_model.wv.most_similar("film"))

[('change', 0.7462551593780518), ('reform', 0.7369738817214966), ('policy', 0.7140024900436401), ('regulation', 0.7135103344917297), ('immigration', 0.7080827951431274), ('argued', 0.7044445872306824), ('welfare', 0.7042416334152222), ('congress', 0.7036283612251282), ('transport', 0.7030194997787476), ('mp', 0.7025432586669922)]
[('fan', 0.7744144797325134), ('fantastic', 0.7215718626976013), ('display', 0.7187051177024841), ('theyve', 0.7135449647903442), ('exciting', 0.7076088190078735), ('english', 0.6999385356903076), ('experience', 0.6951949000358582), ('lover', 0.6948719024658203), ('stuff', 0.6924708485603333), ('osullivan', 0.6921393871307373)]
[('program', 0.8708984851837158), ('tool', 0.8523720502853394), ('window', 0.8344153761863708), ('desktop', 0.8327166438102722), ('microsoft', 0.8253467679023743), ('pirated', 0.8188337087631226), ('microsofts', 0.8074164390563965), ('server', 0.8064063191413879), ('linux', 0.8052049875259399), ('version', 0.8049076795578003)]
[('movie'

a good generalization, focuses on related contextual correct words.

## Document Vectorization
Word -> Documents
to check similarity of nearby vectors

following  common approach:
Document Vector = Average(All Word Vectors)

1 document = average of all vectors in a 1

In [36]:
import numpy as np

def document_vector(doc):
    vectors = [
        w2v_model.wv[word]
        for word in doc
        if word in w2v_model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

In [37]:
doc_vectors = np.array([
    document_vector(doc)
    for doc in news_csv["lemmas"]
])

In [38]:
doc_vectors[1]

array([ 0.0327821 ,  0.10828152,  0.14796044, -0.07286486,  0.08268796,
       -0.27661568,  0.19779254,  0.21592717, -0.04948396, -0.22420649,
       -0.09730912, -0.07891699,  0.02694909, -0.24880365, -0.00320426,
       -0.00490728,  0.2131611 ,  0.02221334,  0.0926797 , -0.17351663,
        0.2643854 , -0.04452675, -0.00989722,  0.03484925, -0.1289062 ,
       -0.20368427,  0.00566414, -0.15282772, -0.23565644, -0.18126978,
        0.05745904,  0.24592543, -0.06577042,  0.03532904, -0.17977554,
        0.09131438, -0.02918496, -0.22074917,  0.03749942, -0.14156464,
        0.31943995, -0.12116314,  0.04554363, -0.18464792,  0.16542219,
        0.0304544 , -0.166123  , -0.0810651 ,  0.19553615,  0.0132932 ,
       -0.03288525, -0.11635108,  0.03274422, -0.06137222, -0.10609393,
        0.03369148,  0.01777626,  0.08458603, -0.13522507, -0.05537147,
        0.08459915, -0.11116728, -0.16619326, -0.01103984, -0.3491895 ,
        0.29836112,  0.2001394 , -0.04004904, -0.28112832,  0.07

In [39]:
print(doc_vectors.shape)

#2225 total documents (rows) as per our dataset & total 100 vectors in each document

(2225, 100)


In [40]:
# finding cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [doc_vectors[1]],
    doc_vectors
)

In [41]:
similarities.argsort()[0][-10:]

array([ 28, 490, 263, 403, 260, 176, 240, 421, 397,   1])

In [42]:
list_last_10 = similarities.argsort()[0][-10:]

for i in list_last_10:
    print(f"{i} : Domain: {news_csv.iloc[i]['category']} & {news_csv.iloc[i]['text']}")

28 : Domain: business & Giant waves damage S Asia economy

Governments, aid agencies, insurers and travel firms are among those counting the cost of the massive earthquake and waves that hammered southern Asia.

The worst-hit areas are Sri Lanka, India, Indonesia and Thailand, with at least 23,000 people killed. Early estimates from the World Bank put the amount of aid needed at about $5bn (Â£2.6bn), similar to the cash offered Central America after Hurricane Mitch. Mitch killed about 10,000 people and caused damage of about $10bn in 1998. World Bank spokesman Damien Milverton told the Wall Street Journal that he expected an aid package of financing and debt relief.

Tourism is a vital part of the economies of the stricken countries, providing jobs for 19 million people in the south east Asian region, according to the World Travel and Tourism Council (WTTC). In the Maldives islands, in the Indian ocean, two-thirds of all jobs depend on tourism.

But the damage covers fishing, farming a

News traversing: Last -> first

oil article
->
other oil articles
->
energy market articles

news top 10

In [43]:
list_top_10 = similarities.argsort()[0][:10]

In [44]:
for i in list_top_10:
  print(f"{i} : Domain: {news_csv.iloc[i]['category']} & {news_csv.iloc[i]['text']}")

1608 : Domain: entertainment & Stars gear up for Bafta ceremony

Film stars from across the globe are preparing to walk the red carpet at this year's Bafta award ceremony.

The 2005 Orange British Academy Film Awards are being held at The Odeon in London's Leicester Square. A host of Hollywood stars, including Cate Blanchett, Leonardo DiCaprio, Keanu Reeves and Richard Gere, are expected to attend Saturday's ceremony. Hosted by Stephen Fry, the glittering ceremony will be broadcast on BBC One at 2010 GMT.

Other actors expected to add to the glamour of the biggest night in UK film are Gael Garcia Bernal, Imelda Staunton, Diane Kruger, Christian Slater, Anjelica Huston, Helen Mirren and former James Bond star Pierce Brosnan. Hollywood blockbuster The Aviator, starring DiCaprio, leads the field with 14 nominations, including best film.

It is up against Eternal Sunshine of the Spotless Mind, Finding Neverland, The Motorcycle Diaries and British film Vera Drake, which has 11 nominations. 

# K-Means Clustering

In [45]:
from sklearn.cluster import KMeans

In [46]:
# Starting with cluste centroids k=5 (same as bbc's output classes)
def clusters(k) :
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    clusters = kmeans.fit_predict(doc_vectors)

    news_csv[f"cluster_k={k}"] = clusters

In [47]:
clusters(5)

In [48]:
news_csv.head()

,category,filename,text,clean_text,tokens,lemmas,lda_text,cluster_k=5
0,business,412.txt,Iran budget seeks state sell-offs\n\nIran's pr...,iran budget seeks state selloffs irans preside...,"[iran, budget, seeks, state, selloffs, irans, ...","[iran, budget, seek, state, selloff, iran, pre...",iran budget seek state selloff iran president ...,4
1,business,311.txt,Stormy year for property insurers\n\nA string ...,stormy property insurers string storms typhoon...,"[stormy, property, insurers, string, storms, t...","[stormy, property, insurer, string, storm, typ...",stormy property insurer string storm typhoon e...,4
2,business,365.txt,Nasdaq planning $100m-share sale\n\nThe owner ...,nasdaq planning mshare sale owner technologydo...,"[nasdaq, planning, mshare, sale, owner, techno...","[nasdaq, planning, mshare, sale, owner, techno...",nasdaq planning mshare sale owner technologydo...,4
3,business,107.txt,Liberian economy starts to grow\n\nThe Liberia...,liberian economy starts grow liberian economy ...,"[liberian, economy, starts, grow, liberian, ec...","[liberian, economy, start, grow, liberian, eco...",liberian economy start grow liberian economy s...,4
4,business,304.txt,Bombardier chief to leave company\n\nShares in...,bombardier chief leave company shares train pl...,"[bombardier, chief, leave, company, shares, tr...","[bombardier, chief, leave, company, share, tra...",bombardier chief leave company share train pla...,0


In [49]:
pd.crosstab(
    news_csv["category"],
    news_csv["cluster_k=5"]
)

cluster_k=5,0,1,2,3,4
category,,,,,
business,217,0,16,8,269
entertainment,51,317,9,0,9
politics,138,3,4,269,3
sport,38,473,0,0,0
tech,38,25,331,1,6


Before removal of extra stopwords:

k=5 was the best cluster


politics: 447/511 are in cluster 2

business: 102/510 are in cluster 2

there maybe government businesses mentioned somewhere related.

In [50]:
clusters(6)

In [51]:
pd.crosstab(
    news_csv["category"],
    news_csv["cluster_k=6"]
)

cluster_k=6,0,1,2,3,4,5
category,,,,,,
business,5,1,221,268,0,15
entertainment,0,250,53,7,68,8
politics,254,0,154,2,3,4
sport,0,74,32,0,405,0
tech,1,10,40,5,21,324


after adding extra stopwords, k=6 give well generalized and more structured output

In [52]:
cluster_docs = news_csv[
    news_csv["cluster_k=6"] == 3
]

from collections import Counter

words = [
    word
    for doc in cluster_docs["lemmas"]
    for word in doc
]

print("business aligned with politics")
Counter(words).most_common(50)

business aligned with politics


[('market', 375),
 ('growth', 365),
 ('economy', 342),
 ('sale', 323),
 ('price', 317),
 ('rate', 279),
 ('economic', 252),
 ('month', 238),
 ('bank', 222),
 ('share', 221),
 ('rise', 215),
 ('profit', 199),
 ('company', 196),
 ('analyst', 195),
 ('country', 192),
 ('dollar', 192),
 ('figure', 186),
 ('government', 185),
 ('firm', 176),
 ('oil', 172),
 ('euro', 169),
 ('world', 159),
 ('however', 155),
 ('china', 155),
 ('quarter', 152),
 ('expected', 150),
 ('cost', 150),
 ('job', 150),
 ('million', 148),
 ('spending', 138),
 ('december', 133),
 ('strong', 132),
 ('rose', 128),
 ('business', 126),
 ('demand', 126),
 ('fall', 125),
 ('consumer', 124),
 ('cut', 123),
 ('high', 122),
 ('interest', 119),
 ('increase', 116),
 ('since', 115),
 ('hit', 115),
 ('trade', 111),
 ('first', 110),
 ('export', 107),
 ('chief', 106),
 ('report', 105),
 ('budget', 104),
 ('economist', 104)]

Yukos was one of the largest and most successful Russian oil and gas companies.

Former majority shareholders launched massive international arbitrations against the Russian government for the expropriation of the company. Courts have ruled that Russia's actions were politically motivated to seize the company's assets.


SO this cluster is aligned in political + business way!

in terms of fraud/legal business issues

In [53]:
cluster_docs = news_csv[
    news_csv["cluster_k=6"] == 2
]

from collections import Counter

words = [
    word
    for doc in cluster_docs["lemmas"]
    for word in doc
]

print("business")
Counter(words).most_common(50)

business


[('company', 516),
 ('government', 476),
 ('firm', 417),
 ('law', 335),
 ('court', 303),
 ('case', 269),
 ('group', 261),
 ('right', 256),
 ('two', 255),
 ('minister', 249),
 ('country', 238),
 ('plan', 223),
 ('deal', 211),
 ('executive', 201),
 ('month', 191),
 ('yukos', 191),
 ('bank', 191),
 ('chief', 190),
 ('share', 190),
 ('state', 190),
 ('report', 184),
 ('world', 179),
 ('police', 176),
 ('business', 172),
 ('home', 172),
 ('decision', 169),
 ('market', 168),
 ('first', 167),
 ('legal', 167),
 ('take', 165),
 ('public', 165),
 ('work', 164),
 ('service', 162),
 ('many', 161),
 ('part', 160),
 ('say', 158),
 ('charge', 155),
 ('foreign', 153),
 ('three', 151),
 ('day', 148),
 ('however', 148),
 ('former', 147),
 ('want', 146),
 ('bid', 141),
 ('trial', 141),
 ('news', 140),
 ('action', 140),
 ('european', 140),
 ('offer', 139),
 ('spokesman', 139)]

In [54]:
cluster_docs = news_csv[
    news_csv["cluster_k=6"] == 0
]

from collections import Counter

words = [
    word
    for doc in cluster_docs["lemmas"]
    for word in doc
]

print("Politics")
Counter(words).most_common(50)

Politics


[('labour', 714),
 ('party', 663),
 ('election', 627),
 ('blair', 523),
 ('tory', 458),
 ('government', 436),
 ('minister', 394),
 ('brown', 350),
 ('tax', 330),
 ('howard', 294),
 ('plan', 267),
 ('prime', 259),
 ('leader', 254),
 ('lord', 236),
 ('chancellor', 234),
 ('public', 216),
 ('issue', 209),
 ('conservative', 201),
 ('general', 198),
 ('campaign', 192),
 ('britain', 192),
 ('country', 170),
 ('tony', 166),
 ('vote', 165),
 ('lib', 158),
 ('claim', 153),
 ('say', 152),
 ('secretary', 149),
 ('michael', 147),
 ('voter', 143),
 ('political', 139),
 ('liberal', 138),
 ('cut', 133),
 ('get', 131),
 ('policy', 131),
 ('kennedy', 128),
 ('democrat', 127),
 ('two', 126),
 ('home', 126),
 ('mp', 126),
 ('council', 121),
 ('take', 117),
 ('want', 117),
 ('think', 116),
 ('budget', 113),
 ('change', 113),
 ('local', 112),
 ('back', 111),
 ('ukip', 111),
 ('way', 110)]

In [55]:
clusters(4)

In [56]:
pd.crosstab(
    news_csv["category"],
    news_csv["cluster_k=4"]
)

cluster_k=4,0,1,2,3
category,,,,
business,112,1,16,381
entertainment,21,335,20,10
politics,400,3,6,8
sport,37,474,0,0
tech,24,25,343,9


In [57]:
clusters(7)

In [58]:
pd.crosstab(
    news_csv["category"],
    news_csv["cluster_k=7"]
)

cluster_k=7,0,1,2,3,4,5,6
category,,,,,,,
business,267,0,217,6,14,0,6
entertainment,7,14,24,0,6,161,174
politics,2,0,142,254,2,0,17
sport,0,311,26,0,0,4,170
tech,6,2,35,1,315,4,38


# Most Important Finding

Word2Vec embeddings revealed that the BBC Business category contains at least two distinct semantic groups: Economic/Market reporting and Corporate/Legal reporting. Increasing the number of clusters from 5 to 6 separated these sub-domains naturally, suggesting that editorial categories do not fully represent the underlying semantic structure of the news corpus.

# Finding silhuette score
to check goodness of datapoints into specific cluster

higher silhuette score = better fit.

In [62]:
doc_vectors.shape

(2225, 100)

In [64]:
from sklearn.metrics import silhouette_score

In [65]:
for k in range(2, 11):
  k_means = KMeans(
      n_clusters = k,
      random_state = 42,
      n_init = 10
  )

  labels = k_means.fit_predict(doc_vectors)

  score = silhouette_score(doc_vectors, labels)
  print(f"Silhouette Score for k={k}: {score}")

Silhouette Score for k=2: 0.269417405128479
Silhouette Score for k=3: 0.23188316822052002
Silhouette Score for k=4: 0.24794228374958038
Silhouette Score for k=5: 0.23652146756649017
Silhouette Score for k=6: 0.19626785814762115
Silhouette Score for k=7: 0.19056151807308197
Silhouette Score for k=8: 0.1874990165233612
Silhouette Score for k=9: 0.18639788031578064
Silhouette Score for k=10: 0.18980436027050018


Conclusion:

Silhouette Score identified k=2 as the most compact clustering solution.

However, qualitative analysis showed that k=5 and k=6 produced substantially more interpretable topic structures corresponding to real-world news domains such as Politics, Technology, Sports, Entertainment, and distinct Business subtopics.

This highlights the difference between geometric cluster quality and semantic interpretability in topic discovery tasks.